# 02 — Silver layer

Clean the bronze data into 3NF with enforced data quality. The technical bronze columns (`_ingested_at`, `_source`, `_batch_id`) are not carried over.

| what | how |
|---|---|
| types | explicit `CAST` to the silver type |
| no missing values | every column `NOT NULL` (enforced by Delta) |
| business rules | `CHECK` on supply cost, available quantity, retail price (enforced by Delta) |
| keys | `PRIMARY KEY` / `FOREIGN KEY`, including the two-column key `lineitem (l_partkey, l_suppkey) -> partsupp (ps_partkey, ps_suppkey)`; informational only, so verified in the Validation section |
| 3NF | `part` is split into `silver_part` and `silver_brand` (see the 3NF section) |
| bad supply cost | rows go to `silver_partsupp_quarantine`, their line items to `silver_lineitem_quarantine` |

Source: `<catalog>.<schema>.bronze_<table>` (8 tables). Target: `<catalog>.<schema>.silver_<table>` (9 tables) plus 2 quarantine tables.

In [0]:
import re

_user = spark.sql("SELECT current_user()").first()[0]
_default_schema = "lakehouse_" + re.sub(r"\W+", "_", _user.split("@")[0]).lower()

dbutils.widgets.text("target_catalog", "workspace", "Target catalog")
dbutils.widgets.text("target_schema", _default_schema, "Target schema")

TARGET_CATALOG = dbutils.widgets.get("target_catalog")
TARGET_SCHEMA = dbutils.widgets.get("target_schema")


def bronze(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_{table}"


def silver(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.silver_{table}"


print(f"Read from : {TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_*")
print(f"Write to  : {TARGET_CATALOG}.{TARGET_SCHEMA}.silver_*")

### Table definitions

In [0]:
COLUMNS = {
    "region": [("r_regionkey", "BIGINT"), ("r_name", "STRING"), ("r_comment", "STRING")],
    "nation": [("n_nationkey", "BIGINT"), ("n_name", "STRING"), ("n_regionkey", "BIGINT"), ("n_comment", "STRING")],
    "supplier": [("s_suppkey", "BIGINT"), ("s_name", "STRING"), ("s_address", "STRING"),
                 ("s_nationkey", "BIGINT"), ("s_phone", "STRING"), ("s_acctbal", "DECIMAL(18,2)"),
                 ("s_comment", "STRING")],
    "customer": [("c_custkey", "BIGINT"), ("c_name", "STRING"), ("c_address", "STRING"),
                 ("c_nationkey", "BIGINT"), ("c_phone", "STRING"), ("c_acctbal", "DECIMAL(18,2)"),
                 ("c_mktsegment", "STRING"), ("c_comment", "STRING")],
    "brand": [("p_brand", "STRING"), ("p_mfgr", "STRING")],
    "part": [("p_partkey", "BIGINT"), ("p_name", "STRING"), ("p_brand", "STRING"),
             ("p_type", "STRING"), ("p_size", "INT"), ("p_container", "STRING"),
             ("p_retailprice", "DECIMAL(18,2)"), ("p_comment", "STRING")],
    "partsupp": [("ps_partkey", "BIGINT"), ("ps_suppkey", "BIGINT"), ("ps_availqty", "INT"),
                 ("ps_supplycost", "DECIMAL(18,2)"), ("ps_comment", "STRING")],
    "orders": [("o_orderkey", "BIGINT"), ("o_custkey", "BIGINT"), ("o_orderstatus", "STRING"),
               ("o_totalprice", "DECIMAL(18,2)"), ("o_orderdate", "DATE"), ("o_orderpriority", "STRING"),
               ("o_clerk", "STRING"), ("o_shippriority", "INT"), ("o_comment", "STRING")],
    "lineitem": [("l_orderkey", "BIGINT"), ("l_partkey", "BIGINT"), ("l_suppkey", "BIGINT"),
                 ("l_linenumber", "INT"), ("l_quantity", "DECIMAL(18,2)"),
                 ("l_extendedprice", "DECIMAL(18,2)"), ("l_discount", "DECIMAL(18,2)"),
                 ("l_tax", "DECIMAL(18,2)"), ("l_returnflag", "STRING"), ("l_linestatus", "STRING"),
                 ("l_shipdate", "DATE"), ("l_commitdate", "DATE"), ("l_receiptdate", "DATE"),
                 ("l_shipinstruct", "STRING"), ("l_shipmode", "STRING"), ("l_comment", "STRING")],
}

CONSTRAINTS = {
    "region": [
        "CONSTRAINT pk_silver_region PRIMARY KEY (r_regionkey)",
    ],
    "nation": [
        "CONSTRAINT pk_silver_nation PRIMARY KEY (n_nationkey)",
        f"CONSTRAINT fk_silver_nation_region FOREIGN KEY (n_regionkey) REFERENCES {silver('region')} (r_regionkey)",
    ],
    "supplier": [
        "CONSTRAINT pk_silver_supplier PRIMARY KEY (s_suppkey)",
        f"CONSTRAINT fk_silver_supplier_nation FOREIGN KEY (s_nationkey) REFERENCES {silver('nation')} (n_nationkey)",
    ],
    "customer": [
        "CONSTRAINT pk_silver_customer PRIMARY KEY (c_custkey)",
        f"CONSTRAINT fk_silver_customer_nation FOREIGN KEY (c_nationkey) REFERENCES {silver('nation')} (n_nationkey)",
    ],
    "brand": [
        "CONSTRAINT pk_silver_brand PRIMARY KEY (p_brand)",
    ],
    "part": [
        "CONSTRAINT pk_silver_part PRIMARY KEY (p_partkey)",
        f"CONSTRAINT fk_silver_part_brand FOREIGN KEY (p_brand) REFERENCES {silver('brand')} (p_brand)",
    ],
    "partsupp": [
        "CONSTRAINT pk_silver_partsupp PRIMARY KEY (ps_partkey, ps_suppkey)",
        f"CONSTRAINT fk_silver_partsupp_part FOREIGN KEY (ps_partkey) REFERENCES {silver('part')} (p_partkey)",
        f"CONSTRAINT fk_silver_partsupp_supplier FOREIGN KEY (ps_suppkey) REFERENCES {silver('supplier')} (s_suppkey)",
    ],
    "orders": [
        "CONSTRAINT pk_silver_orders PRIMARY KEY (o_orderkey)",
        f"CONSTRAINT fk_silver_orders_customer FOREIGN KEY (o_custkey) REFERENCES {silver('customer')} (c_custkey)",
    ],
    "lineitem": [
        "CONSTRAINT pk_silver_lineitem PRIMARY KEY (l_orderkey, l_linenumber)",
        f"CONSTRAINT fk_silver_lineitem_orders FOREIGN KEY (l_orderkey) REFERENCES {silver('orders')} (o_orderkey)",
        f"CONSTRAINT fk_silver_lineitem_partsupp FOREIGN KEY (l_partkey, l_suppkey) "
        f"REFERENCES {silver('partsupp')} (ps_partkey, ps_suppkey)",
    ],
}

CREATE_ORDER = ["region", "nation", "supplier", "customer", "brand", "part", "partsupp", "orders", "lineitem"]

### Reset

In [0]:
for t in reversed(CREATE_ORDER):
    spark.sql(f"DROP TABLE IF EXISTS {silver(t)}")
for q in ("lineitem_quarantine", "partsupp_quarantine"):
    spark.sql(f"DROP TABLE IF EXISTS {silver(q)}")

### Quarantine

In [0]:
ps_cols = ", ".join(f"ps.{n}" for n, _ in COLUMNS["partsupp"])
spark.sql(f"""
    CREATE TABLE {silver('partsupp_quarantine')} AS
    SELECT {ps_cols},
           CASE WHEN ps.ps_supplycost <= 0 THEN 'supply cost is not positive'
                ELSE 'supply cost is above part retail price' END AS reason
    FROM {bronze('partsupp')} ps
    JOIN {bronze('part')} p ON p.p_partkey = ps.ps_partkey
    WHERE ps.ps_supplycost <= 0 OR ps.ps_supplycost > p.p_retailprice
""")

li_cols = ", ".join(f"l.{n}" for n, _ in COLUMNS["lineitem"])
spark.sql(f"""
    CREATE TABLE {silver('lineitem_quarantine')} AS
    SELECT {li_cols}, 'part-supplier pair is in partsupp_quarantine' AS reason
    FROM {bronze('lineitem')} l
    LEFT SEMI JOIN {silver('partsupp_quarantine')} q
      ON l.l_partkey = q.ps_partkey AND l.l_suppkey = q.ps_suppkey
""")

### Load

In [0]:
FROM_CLAUSE = {
    "brand": f"(SELECT DISTINCT p_brand, p_mfgr FROM {bronze('part')}) b",
    "partsupp": f"{bronze('partsupp')} ps LEFT ANTI JOIN {silver('partsupp_quarantine')} q "
                f"ON ps.ps_partkey = q.ps_partkey AND ps.ps_suppkey = q.ps_suppkey",
    "lineitem": f"{bronze('lineitem')} l LEFT ANTI JOIN {silver('partsupp_quarantine')} q "
                f"ON l.l_partkey = q.ps_partkey AND l.l_suppkey = q.ps_suppkey",
}

for t in CREATE_ORDER:
    cols = ",\n  ".join(f"{name} {typ} NOT NULL" for name, typ in COLUMNS[t])
    cons = ",\n  ".join(CONSTRAINTS[t])
    spark.sql(f"CREATE TABLE {silver(t)} (\n  {cols},\n  {cons}\n)")

    select = ", ".join(f"CAST({name} AS {typ}) AS {name}" for name, typ in COLUMNS[t])
    source = FROM_CLAUSE.get(t, bronze(t))
    spark.sql(f"INSERT INTO {silver(t)} SELECT {select} FROM {source}")

    spark.sql(f"COMMENT ON TABLE {silver(t)} IS 'Silver: {t} (3NF, typed, constraints).'")
    print("loaded", t)

### Check: bronze rows = silver rows + quarantined rows

In [0]:
quarantined = {
    "partsupp": spark.table(silver("partsupp_quarantine")).count(),
    "lineitem": spark.table(silver("lineitem_quarantine")).count(),
}

rows = []
for t in CREATE_ORDER:
    if t == "brand":
        continue  
    b = spark.table(bronze(t)).count()
    s = spark.table(silver(t)).count()
    q = quarantined.get(t, 0)
    rows.append((t, b, s, q, b == s + q))

report = spark.createDataFrame(
    rows, "table_name STRING, bronze_rows LONG, silver_rows LONG, quarantined_rows LONG, matches BOOLEAN"
)
display(report)

mismatched = [r[0] for r in rows if not r[4]]
assert not mismatched, f"Row count mismatch between bronze and silver: {mismatched}"

brands_bronze = spark.sql(f"SELECT COUNT(DISTINCT p_brand) FROM {bronze('part')}").first()[0]
brands_silver = spark.table(silver("brand")).count()
assert brands_bronze == brands_silver, f"Brand count mismatch: {brands_bronze} vs {brands_silver}"
print("OK: no rows were lost between bronze and silver.")

### Validation

In [0]:
CHECKS = [
    ("lineitem (partkey, suppkey) pair exists in partsupp",
     f"""SELECT COUNT(*) FROM {silver('lineitem')} l
         LEFT ANTI JOIN {silver('partsupp')} ps
           ON l.l_partkey = ps.ps_partkey AND l.l_suppkey = ps.ps_suppkey"""),

    ("supplier resolves to a valid nation",
     f"""SELECT COUNT(*) FROM {silver('supplier')} s
         LEFT ANTI JOIN {silver('nation')} n ON s.s_nationkey = n.n_nationkey"""),

    ("nation resolves to a valid region",
     f"""SELECT COUNT(*) FROM {silver('nation')} n
         LEFT ANTI JOIN {silver('region')} r ON n.n_regionkey = r.r_regionkey"""),

    ("each brand has exactly one manufacturer (needed for the brand split)",
     f"""SELECT COUNT(*) FROM (
             SELECT p_brand FROM {bronze('part')}
             GROUP BY p_brand HAVING COUNT(DISTINCT p_mfgr) > 1)"""),

    ("part resolves to a valid brand",
     f"""SELECT COUNT(*) FROM {silver('part')} p
         LEFT ANTI JOIN {silver('brand')} b ON p.p_brand = b.p_brand"""),

    ("supply cost is positive",
     f"SELECT COUNT(*) FROM {silver('partsupp')} WHERE ps_supplycost <= 0"),

    ("supply cost is not above the part retail price",
     f"""SELECT COUNT(*) FROM {silver('partsupp')} ps
         JOIN {silver('part')} p ON p.p_partkey = ps.ps_partkey
         WHERE ps.ps_supplycost > p.p_retailprice"""),
]

results = []
for name, query in CHECKS:
    bad = spark.sql(query).first()[0]
    results.append((name, bad, "PASS" if bad == 0 else "FAIL"))

display(spark.createDataFrame(results, "check_name STRING, violations LONG, status STRING"))

failed = [r[0] for r in results if r[2] == "FAIL"]
assert not failed, f"Validation failed: {failed}"
print("OK: all validation checks passed.")

In [0]:
display(spark.sql(f"""
    SELECT 'silver_partsupp_quarantine' AS table_name, reason, COUNT(*) AS n_rows
    FROM {silver('partsupp_quarantine')} GROUP BY reason
    UNION ALL
    SELECT 'silver_lineitem_quarantine' AS table_name, reason, COUNT(*) AS n_rows
    FROM {silver('lineitem_quarantine')} GROUP BY reason
"""))